# SmolLM Inference 

In [ ]:
https://huggingface.co/docs/transformers/main_classes/pipelines#pipeline-batching

In [37]:
import torch
import time
import functools
from typing import Callable, Any

def monitor_gpu_inference(func: Callable) -> Callable:
    """Decorator to monitor GPU usage and timing during inference"""
    
    @functools.wraps(func)
    def wrapper(*args, **kwargs) -> Any:
        # Clear cache and get initial state
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        
        initial_memory = torch.cuda.memory_allocated()
        start_time = time.time()
        
        try:
            # Execute the function
            result = func(*args, **kwargs)
            
            # Calculate metrics
            end_time = time.time()
            final_memory = torch.cuda.memory_allocated()
            peak_memory = torch.cuda.max_memory_allocated()
            
            # Print results
            print(f"🚀 Function: {func.__name__}")
            print(f"⏱️  Execution Time: {end_time - start_time:.3f} seconds")
            print(f"💾 Memory Used: {(final_memory - initial_memory) / 1024**2:.2f} MB")
            print(f"📊 Peak Memory: {peak_memory / 1024**2:.2f} MB")
            print(f"🎯 Memory Efficiency: {(final_memory / peak_memory * 100):.1f}%")
            print("-" * 50)
            
            return result
            
        except Exception as e:
            print(f"❌ Error in {func.__name__}: {e}")
            raise
            
    return wrapper

In [41]:
# Batch inference
@monitor_gpu_inference
def batch_inference(dataset, n, system_prompt, sample_type='test', batch_size=10):

    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer,
                    torch_dtype=torch.float16)
    pipe.model.eval()  # Set to evaluation mode
    return json.dumps(pipe(dataset[sample_type][:n]['messages'], batch_size=batch_size), indent=4)

In [ ]:
n = 10000

system_prompt_summarize = "Provide a concise, objective summary of the input text in up to three sentences, focusing on key actions and intentions without using second or third person pronouns."
results = batch_inference(evaluation_dataset, n, system_prompt_summarize)

Device set to use cuda
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, pleas

In [ ]:
# Use CPU instead

import torch
from transformers import pipeline

# Clear cache before starting
torch.cuda.empty_cache()

# Setup with memory optimization
pipe = pipeline(
    "text-generation",
    model="your-model",
    device=0,
    torch_dtype=torch.float16,
    model_kwargs={
        "low_cpu_mem_usage": True,
        "use_cache": True,  # Enable KV cache for faster generation
    }
)

# Optimize generation parameters
generation_config = {
    "max_new_tokens": 256,  # Instead of max_length
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.9,
    "repetition_penalty": 1.1,
    "pad_token_id": pipe.tokenizer.eos_token_id,
    "use_cache": True,
}

# Generate with optimized settings
result = pipe("Your prompt", **generation_config)